In [1]:
%pip install numpy
%pip install opencv-python
%pip install scipy

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import cv2
from scipy.ndimage import convolve, median_filter

In [3]:
# show img window
def showimage(original_img, filtered_img, filter_name, imgsize = (512, 512)):
    """     Display the original and filtered images side by side.
            Parameters
            ----------
                original_img : numpy.ndarray
                The original input image.

                filtered_img : numpy.ndarray
                The processed or filtered image. It will be converted to uint8 before display.

                imgsize : tuple of int, optional
                The desired size (width, height) to which both images will be resized.
                Default is (512, 512).
    """
    
    filtered_img = filtered_img.astype(np.uint8)  # change pixel type
    filtered_img = cv2.resize(filtered_img, imgsize, interpolation=cv2.INTER_AREA)
    original_img = cv2.resize(original_img, imgsize, interpolation=cv2.INTER_AREA)
    two_imgs = np.hstack((original_img, filtered_img))
    cv2.imshow(f'Apply {filter_name} filter', two_imgs)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

In [4]:
# read image for testing
img = cv2.imread("./img.jpg")

# remove the RGB channel
img = np.mean(img, axis=2).astype(np.uint8)

# check negative and exceeded values
img = np.abs(img)
img[img > 255] = 255

print(f"""
      Shape: {np.shape(img)}
      dtype: {img.dtype}
      MAX val: {np.max(img)}
      MIN val: {np.min(img)}
      """)


      Shape: (1204, 1880)
      dtype: uint8
      MAX val: 255
      MIN val: 0
      


In [5]:
# 1.1 Negative
neg_img = 255 - img
showimage(img, neg_img, "Negative")

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in "/home/sherif/data/Engineering/My faculties/Faculty of Engineering/4/The second semester/.venv/lib/python3.14/site-packages/cv2/qt/plugins"
QFontDatabase: Cannot find font directory /home/sherif/data/Engineering/My faculties/Faculty of Engineering/4/The second semester/.venv/lib/python3.14/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/sherif/data/Engineering/My faculties/Faculty of Engineering/4/The second semester/.venv/lib/python3.14/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/sherif/data/Engineering/My faculties/Faculty of Engineering/4/The second semester/.venv/lib/python3.14/site-packages/cv2/qt/fonts.
Note that

In [6]:
# 1.2 Threshold
threshold_val = 255 // 2 + 1
threshold_img = np.where(img > threshold_val, 255, 0)
showimage(img, threshold_img, "Threshold")

In [7]:
# 1.3 Logarithmic
normed_img = img.astype(np.float32) / 255
log_img = np.log1p(1 + normed_img)
log_img = (log_img / np.max(log_img)) * 255
showimage(img, log_img, "Logarithmic")

In [8]:
# 1.4 Power law transformation
gamma = 2.2
normed_img = img.astype(np.float32) / 255
Power_img = np.power(normed_img, gamma)
Power_img = (Power_img / np.max(Power_img)) * 255
showimage(img, Power_img, "Power law transformation")

In [9]:
# 1.5 Grey Level Slicing
a, b = 100, 150
graylvl_img = np.where((img >= a) & (img <= b), 255, 0)
showimage(img, graylvl_img, "Grey Level Slicing")

In [10]:
# 1.6 Bit Plane Slicing
bitpln_img = img & 0b11111000   # Zero the 3 (LSB) bits
showimage(img, bitpln_img, "Bit Plane Slicing")

In [11]:
# 2.1 Smoothing Filter (Simple Averaging / Box Filter)
# 2.3 Average Filter (General N×N)
n = 15
kernel_avg = np.ones((n,n)) / n**2
box_img = convolve(img, kernel_avg, mode="reflect")
showimage(img, box_img, "Average Filter")

In [12]:
# 2.2 Weighted Smoothing Filter (Weighted Average)
kernel_weg = np.array([
    [1, 2, 1],
    [2, 4, 2],
    [1, 2, 1]
]).astype(float)

kernel_weg = np.tile(kernel_weg, 12) # repeat kernel_weg
kernel_weg /= kernel_weg.sum()
weg_img = convolve(img, kernel_weg, mode="reflect")
showimage(img, weg_img, "Weighted Smoothing")

In [13]:
# 2.4 Median Filter
median_img = median_filter(img, size=7, mode="reflect")
showimage(img, median_img, "Median")

In [14]:
# 2.5 Laplacian Filter (2nd Derivative Sharpening)

# Standard kernel (4-connectivity):
w = np.array([
    [0 , -1,  0],
    [-1,  5, -1],
    [0 , -1,  0]
])

lap_img = convolve(img.astype(np.float32), w, mode="reflect")
showimage(img, lap_img, "Laplacian")


In [15]:
# 2.6 Sobel Filter (1st Derivative Edge Detection)
Sx = np.array([
    [-1, -2, -1],
    [0 ,  0,  0],
    [1 ,  2,  1]
])

Sy = np.array([
    [-1,  0,  1],
    [-2,  0,  2],
    [-1,  0,  1]
])

Gx = convolve(img.astype(np.float32), Sx, mode="reflect")
Gy = convolve(img.astype(np.float32), Sy, mode="reflect")
sobel_img = np.sqrt(Gx**2 + Gy**2)
sobel_img = (sobel_img / np.max(sobel_img)) * 255
print(np.max(sobel_img))


showimage(img, sobel_img, "Sobel")


255.0


In [16]:
def DFT(temp_img):
    img2DFT = temp_img.astype(np.float32)
    # 2D DFT
    img2DFT = np.fft.fft2(img2DFT)
    # shift zero-frequency to center
    img2DFT = np.fft.fftshift(img2DFT)
    return img2DFT

def IDFT(temp_img):
    img2IDFT = np.fft.ifftshift(temp_img)
    img2IDFT = np.fft.ifft2(img2IDFT)
    img2IDFT = np.abs(img2IDFT)
    img2IDFT = (img2IDFT / np.max(img2IDFT)) * 255
    img2IDFT = img2IDFT.astype(np.uint8)
    return img2IDFT

def complex2normal(complex_img):
    normal_img = np.log1p(np.abs(complex_img))   # normal_img + log scaling
    # normalize for display
    normal_img = (normal_img / np.max(normal_img)) * 255
    normal_img = normal_img.astype(np.uint8)
    return normal_img

In [38]:
# 3.1.1 Ideal Low Pass Filter (ILPF)
complex_img = DFT(img)

cnt_row = img.shape[0] // 2
cnt_col = img.shape[1] // 2
radius = 30

u = np.arange(stop=img.shape[0])
v = np.arange(stop=img.shape[1])

U, V = np.meshgrid(u, v, indexing='ij')

dist2cnt = np.sqrt((U - cnt_row)**2 + (V - cnt_col)**2)
LPF = (dist2cnt <= radius).astype(np.float32)

ideal_img = complex_img * LPF   # apply LPF

ideal_img = IDFT(ideal_img)

showimage(img, ideal_img, "Ideal Low Pass")

In [39]:
# 3.1.2 Ideal High Pass Filter (IHPF)
complex_img = DFT(img)

cnt_row = img.shape[0] // 2
cnt_col = img.shape[1] // 2
radius = 135

u = np.arange(stop=img.shape[0])
v = np.arange(stop=img.shape[1])

U, V = np.meshgrid(u, v, indexing='ij')

dist2cnt = np.sqrt((U - cnt_row)**2 + (V - cnt_col)**2)
HPF = (dist2cnt > radius).astype(np.float32)

ideal_img = complex_img * HPF   # apply HPF

ideal_img = IDFT(ideal_img)

showimage(img, ideal_img, "Ideal High Pass")

In [48]:
# 3.2.1 Butterworth Low Pass Filter
complex_img = DFT(img)

cnt_row = img.shape[0] // 2
cnt_col = img.shape[1] // 2
radius = 13
ord = 2

u = np.arange(stop=img.shape[0])
v = np.arange(stop=img.shape[1])

U, V = np.meshgrid(u, v, indexing='ij')

dist2cnt = np.sqrt((U - cnt_row)**2 + (V - cnt_col)**2)
BLPF = 1 / (1 + (dist2cnt / radius)**(2*ord))

BLPF_img = complex_img * BLPF   # apply BLPF

BLPF_img = IDFT(BLPF_img)

showimage(img, BLPF_img, "Butterworth Low Pass")

In [50]:
# 3.2.1 Butterworth High Pass Filter
complex_img = DFT(img)

cnt_row = img.shape[0] // 2
cnt_col = img.shape[1] // 2
radius = 37
ord = 5

u = np.arange(stop=img.shape[0])
v = np.arange(stop=img.shape[1])

U, V = np.meshgrid(u, v, indexing='ij')

dist2cnt = np.sqrt((U - cnt_row)**2 + (V - cnt_col)**2)

# avoid division by zero
dist2cnt = np.where(dist2cnt == 0, 1e-6, dist2cnt)
BHPF = 1 / (1 + (radius / dist2cnt)**(2*ord))

BHPF_img = complex_img * BHPF   # apply BHPF

BHPF_img = IDFT(BHPF_img)

showimage(img, BHPF_img, "Butterworth High Pass")